<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# GeoPandas I: Spatial Data Model

### CP101: Introduction to Urban Data Analytics

**Week 6, Wednesday** · GeoPandas I

**Estimated time:** 60–75 minutes with instructor walkthroughs. The inset map section at the end is optional.

**Not graded, not submitted.** Run cells in order with `Shift + Enter`. Complete each **Try It Out**; the Otter cells offer optional feedback on your results.

[GeoPandas user guide](https://geopandas.org/en/stable/docs/user_guide.html) · [NYC Building Footprints metadata](https://github.com/CityOfNewYork/nyc-geo-metadata/blob/main/Metadata/Metadata_BuildingFootprints.md)

In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("CP101_W06_S2_geopandas.ipynb")

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Learning Objectives

<hr style="border: 1px solid #fdb515;" />

Earlier this week you cleaned and joined tables with pandas. This session adds a geometry column to that workflow. By the end, you should be able to:

1. Explain how a `GeoDataFrame` extends a `DataFrame`, and what its geometry column holds.
2. Read vector data from common file formats, filter it at read time, and write it back to disk.
3. Apply basic geometric operations to individual geometries.
4. Explain how a coordinate reference system determines the units of a measurement, and reproject data before measuring.
5. Produce static and interactive maps, and check that each one represents its variable correctly.

The optional extension adds inset maps.

### Data for this notebook

Every file is already in `data/raw/`; no download is needed.

| File | What it holds |
|---|---|
| `ne_110m_admin_0_countries.parquet` | Country boundaries (Natural Earth), as GeoParquet |
| `ne_110m_admin_0_countries/` | The same boundaries, as a Shapefile |
| `Bicycle_Counters_20250301.csv` | NYC bicycle counter locations, as latitude and longitude columns |
| `coney_island_ed.parquet` | Election district polygons around Coney Island |
| `manhattan_bldgfp_lower.geojson` | NYC building footprints for part of Manhattan |
| `nyc_neighborhoods_small.parquet` | NYC neighborhood boundaries (optional section) |

Treat `data/raw/` as read-only: load files into variables and work with those. Anything this notebook writes goes to `outputs/`.

### What GeoPandas adds

`GeoPandas` provides an easy-to-use interface to vector datasets. It combines the capabilities of `pandas` with the geometry handling functionality of `shapely`.

The main data structures in GeoPandas are `GeoDataFrame` and `GeoSeries`. They extend `pandas.DataFrame` and `pandas.Series`, so all pandas operations you already know continue to work.

A `GeoDataFrame` contains a tabular geospatial dataset:

- It has a **geometry column** that holds the geometry information (or features in GeoJSON).
- All other columns are **attributes** (or properties in GeoJSON) that describe each geometry.

A `GeoDataFrame` is just like a pandas `DataFrame`, but with additional functionality for spatial work:

- A `.geometry` attribute that always returns the active geometry column as a `GeoSeries`, regardless of what the column is named.
- Extra methods for spatial operations (area, distance, buffer, intersection, ...) covered later in this session.

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Setup

<hr style="border: 1px solid #fdb515;" />

In [ ]:
!pip install "geopandas[all]"

In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

# Course palette, used for every map in this notebook.
BERKELEY_BLUE = "#003262"
BERKELEY_GOLD = "#fdb515"

# Files this notebook writes go here, never into data/raw/.
OUTPUTS = Path("outputs")
OUTPUTS.mkdir(exist_ok=True)

print("pandas version", pd.__version__)
print("geopandas version", gpd.__version__)

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## File I/O

<hr style="border: 1px solid #fdb515;" />

Geospatial data comes in many formats. GeoPandas reads most of them with a single function, `gpd.read_file()`. Below we cover the formats you will encounter most often.

### Reading a Parquet file

Parquet is a columnar binary format built for large datasets. GeoParquet extends it with geometry support, and both use the `.parquet` extension. It loads faster and compresses better than a Shapefile or GeoJSON, and it is widely used in the Python geospatial community. GeoPandas reads it with its own function, `gpd.read_parquet("<file path>")`.

In [ ]:
countries = gpd.read_parquet("data/raw/ne_110m_admin_0_countries.parquet")
countries.head(3)

In [ ]:
# .geometry returns the active geometry column as a GeoSeries
# Notice the geometry type printed at the start of each row
countries.geometry

In [ ]:
# GeoDataFrames often have a coordinate reference system (crs)
countries.crs

In [ ]:
# .plot() draws the geometry column with matplotlib under the hood.
ax = countries.plot(
    color=BERKELEY_BLUE,
    edgecolor="white",
    linewidth=0.3,
    figsize=(11, 5.5),
)
ax.set_title("Natural Earth country boundaries", loc="left")
ax.set_axis_off()
plt.tight_layout()
plt.show()

### Reading a Shapefile

A Shapefile is a geospatial vector format introduced by **Esri**. It stores points, lines, or polygons along with attribute data. It is made up of multiple related files and most of them should be in the same folder for a shapefile to work:

- `.shp`: geometry
- `.shx`: index for fast access
- `.dbf`: attribute data in tabular format
- `.prj` (optional): coordinate system definition

When reading a Shapefile with GeoPandas, you can specify either the folder or the `.shp` file directly. GeoPandas loads the other files automatically if they are in the same directory.

Shapefiles have known limitations: 2 GB file size limit, restricted column name lengths, and slower performance on large datasets. More efficient alternatives include GeoJSON (web-friendly), GeoParquet (large datasets), and GeoPackage (single-file, supported by QGIS and ArcGIS).

In [ ]:
# Pass the folder: GeoPandas finds and reads the .shp file inside it.
countries_shp = gpd.read_file("data/raw/ne_110m_admin_0_countries")
countries_shp.head(3)

In [ ]:
# Or read the .shp file directly.
# If a folder holds several shapefiles, name the one you want.
# countries_shp = gpd.read_file("data/raw/ne_110m_admin_0_countries/ne_110m_admin_0_countries.shp")

### Reading from a URL

GeoPandas can load spatial files directly from a web URL. The format is inferred from the URL.

In [ ]:
url = "https://d2ad6b4ur7yvpq.cloudfront.net/naturalearth-3.3.0/ne_110m_land.geojson"
countries_web = gpd.read_file(url)
countries_web.head(3)

### Reading from a CSV file

A lot of geospatial point data, such as incident-level records, arrives as a `.csv`. A CSV with longitude and latitude columns is implicitly spatial. To turn one into a `GeoDataFrame`:

1. Load the CSV into a pandas DataFrame.
2. Identify the longitude and latitude columns and confirm their CRS.
3. Use `gpd.points_from_xy()` to build a geometry column. **Order matters: longitude (x) first, latitude (y) second.**
4. Construct a `GeoDataFrame` from the DataFrame, the geometry, and the CRS.

In [ ]:
# Step 1: load the CSV with pandas
df_csv = pd.read_csv("data/raw/Bicycle_Counters_20250301.csv")

# Step 2: check the column names
df_csv.columns

In [ ]:
# Step 3: create geometry array from longitude and latitude
gdf_geom = gpd.points_from_xy(df_csv['longitude'], df_csv['latitude'])
gdf_geom

In [ ]:
# Step 4: construct the GeoDataFrame
gdf_counter = gpd.GeoDataFrame(
    df_csv,              # the pandas DataFrame
    geometry=gdf_geom,   # the geometry column of shapely objects
    crs=4326,            # longitude and latitude in WGS84
)
gdf_counter.head(3)

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Geometric Operations on Shapely Geometries

<hr style="border: 1px solid #fdb515;" />

The objects stored in the geometry column are `shapely` objects. GeoPandas uses shapely as its geometry engine: the two packages are developed in close coordination.

A `GeoDataFrame` can technically hold multiple geometry columns (for example, one with polygons, one with their centroids), but only one is active at a time. You can always access the active column with `.geometry`, regardless of its name.

In [ ]:
coney_island_ed = gpd.read_parquet("data/raw/coney_island_ed.parquet")
coney_island_ed.head(3)

In [ ]:
# Isolate one geometry to explore shapely operations
long_region = coney_island_ed.loc[23, "geometry"]   # row 23, geometry column
long_region

In [ ]:
# .boundary returns the outline, which is no longer a polygon.
# It is an attribute, not a method, so there are no parentheses.
long_region.boundary

In [ ]:
print(type(long_region.boundary))

### Area, perimeter, envelope, bounding box, centroid

In [ ]:
print("area:     ", long_region.area)

In [ ]:
print("perimeter:", long_region.length)

In [ ]:
# Axis-aligned bounding box
env = long_region.envelope
env

In [ ]:
# Minimum rotated rectangle: aligns with the shape's own orientation
long_region.minimum_rotated_rectangle

In [ ]:
cent = long_region.centroid
cent

### Simplification

`.simplify()` applies the Ramer-Douglas-Peucker (RDP) algorithm. A higher tolerance produces a coarser shape. Why simplify? Think about file size and rendering speed when drawing complex boundaries at a small scale.

In [ ]:
long_region.simplify(100)

In [ ]:
long_region.simplify(600)   # higher tolerance, coarser and less accurate

### Buffer

`.buffer()` expands a geometry by a given distance in the units of the CRS. What are the units here? Check `coney_island_ed.crs` before interpreting the result.

In [ ]:
coney_island_ed.crs

In [ ]:
# What unit is 20 in? Check coney_island_ed.crs before using this for analysis.
cent_buff = cent.buffer(20)
cent_buff

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Loading Data with a Bounding Box Filter

<hr style="border: 1px solid #fdb515;" />

For large datasets, loading everything into memory and then filtering is wasteful. GeoPandas supports pre-filtering at read time using a bounding box or a geometry mask. The bounding box must be in the same CRS as the source file.

In [ ]:
# Coordinates from Google Maps (longitude, latitude order)
bbox = (
    -73.99260908088572, 40.74341052862094,   # southwest corner
    -73.97417702918338, 40.7575719513787,    # northeast corner
)

bldg_fpt = gpd.read_file(
    "data/raw/manhattan_bldgfp_lower.geojson",
    bbox=bbox,
)

Let's look at the data dictionary from NYC’s **Building Footprints** (OTI/DoITT) schema. The authoritative attribute definitions are in NYC’s public metadata for Building Footprints. ([metadata on GitHub](https://github.com/CityOfNewYork/nyc-geo-metadata/blob/main/Metadata/Metadata_BuildingFootprints.md))

### Data dictionary (NYC Building Footprints)

| Column (your GeoDataFrame) | Official meaning                                  | Short notes                                   |
| -------------------------- | ------------------------------------------------- | --------------------------------------------- |
| `name`                     | Building name (limited to commonly known names)   | Not maintained; many nulls.                   |
| `base_bbl`                 | BBL of the tax lot the footprint is physically on | Text-typed; occasional lot mismatch.          |
| `mpluto_bbl`               | BBL for joining to MapPLUTO (billing/condo logic) | Differs for condos; use for PLUTO joins.      |
| `bin`                      | NYC Building Identification Number                | Not always unique (placeholder BINs).         |
| `cnstrct_yr`               | Construction completion year                      | Often missing/0; source varies by era.        |
| `globalid`                 | UUID identifier                                   | Not present in every distribution.            |
| `lststatype`               | Last status/change type                           | What kind of edit happened last.              |
| `feat_code`                | Structure/building type code                      | Categorical code (building, garage, etc.).    |
| `groundelev`               | Lowest ground elevation at footprint              | From LiDAR/photogrammetry; variable accuracy. |
| `geomsource`               | Geometry capture/update method                    | Photogrammetric vs manual digitizing.         |
| `heightroof`               | Roof height above ground                          | Missing common; multiple measurement sources. |
| `lstmoddate`               | Last modified date                                | Timestamp of last edit.                       |
| `doitt_id`                 | OTI/DoITT stable unique ID                        | Best primary key.                             |
| `shape_area`               | System-generated area                             | Don’t use; recompute in projected CRS.        |
| `shape_len`                | System-generated length                           | Don’t use; recompute in projected CRS.        |
| `geometry`                 | Building footprint polygon                        | Preferred local CRS is EPSG:2263. Exports use EPSG:4326 or EPSG:3857.  |



### Two quick sanity checks that usually matter in analysis

1. **If you plan to compute area/perimeter:** ignore `shape_area` / `shape_len` and recompute after projecting to a suitable CRS (commonly EPSG:2263 for NYC work, or an equal-area CRS if you’re doing area-sensitive statistics).
2. **If you need a unique building key:** use `doitt_id` (not BIN).

### Inspect the data

In [ ]:
bldg_fpt.crs

In [ ]:
bldg_fpt.info()

The bounding box covers Midtown Manhattan. Suppose we want a map of building heights. `heightroof` records:

> The height of the roof above the ground elevation, not height above sea level.

<hr style="border: 1px solid #fdb515;" />

### ✏️ Try It Out 1: Why does column type matter for mapping?

Before plotting `heightroof`, think about what kind of variable building height is.

**Step 1.** Look at the values and the dtype.

In [ ]:
print(bldg_fpt['heightroof'].dtype)
bldg_fpt['heightroof'].head(10)

**Step 2.** Answer these before running the plot:

- Is building height a continuous or categorical variable?
- When GeoPandas colors a map by a column, how does it decide whether to use a continuous color gradient or a categorical palette?
- Based on the dtype you see above, what do you expect the map and legend to look like?

In [ ]:
# Run this and compare the output to your prediction.
# Left deliberately unstyled: what matters here is the legend GeoPandas chooses.
bldg_fpt.plot("heightroof", legend=True)

<hr style="border: 1px solid #fdb515;" />

**Step 3.** What did you get? Does the legend show what you expected?

If it is not what you wanted, decide what needs to change and why. Fix the column in place in `bldg_fpt`, then run the plot below it.

In [ ]:
bldg_fpt["heightroof"] = ...

In [ ]:
grader.check("ca06s2_heightroof")

In [ ]:
ax = bldg_fpt.plot(
    "heightroof",
    cmap="cividis",
    legend=True,
    figsize=(9, 9),
    legend_kwds={"label": "Roof height above ground (ft)", "shrink": 0.5},
)
ax.set_title("Midtown Manhattan building footprints by roof height", loc="left")
ax.set_axis_off()
plt.tight_layout()
plt.show()

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Writing Files

<hr style="border: 1px solid #fdb515;" />

To write a `GeoDataFrame` to a file, use `.to_file()`. The format is inferred from the file extension, or you can name it with `driver=`.

In [ ]:
bldg_fpt.to_file(OUTPUTS / "bldg_flt.geojson", driver="GeoJSON")

In [ ]:
# Parquet: faster and smaller for large datasets
bldg_fpt.to_parquet(OUTPUTS / "bldg_flt.parquet")

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Maps in GeoPandas

<hr style="border: 1px solid #fdb515;" />

GeoPandas produces two kinds of maps: static images with `.plot()` and interactive maps with `.explore()`.

### Static maps

Call `.plot()` on any `GeoDataFrame`. Pass a column name as the first argument to color by that column's values.

In [ ]:
ax = bldg_fpt.plot(
    "heightroof",
    cmap="cividis",
    legend=True,
    figsize=(9, 9),
    legend_kwds={"label": "Roof height above ground (ft)", "shrink": 0.5},
)
ax.set_title("Midtown Manhattan building footprints by roof height", loc="left")
ax.set_axis_off()
plt.tight_layout()
plt.savefig(OUTPUTS / "midtown_building_height.png", dpi=300)
plt.show()

### Interactive maps

`.explore()` works like `.plot()` but returns an interactive map built on Leaflet. `tiles=` sets the basemap, and `popup=` lists the columns shown when you click a building.

In [ ]:
m = bldg_fpt.explore(
    "heightroof",
    tiles="Stadia.AlidadeSmooth",
    cmap="cividis",
    tooltip=False,
    popup=["name", "heightroof"],
    legend=True,
)
m

In [ ]:
m.save(OUTPUTS / "bldg_fpt_heightroof_map.html")

<!-- BEGIN QUESTION -->

<hr style="border: 1px solid #fdb515;" />

### ✏️ Try It Out 2: Change the basemap

Redraw the map above on the `"Stadia.AlidadeSmoothDark"` basemap. Which buildings stand out more on a dark background, and why?

In [ ]:
m = bldg_fpt.explore(
    "heightroof",
    tiles=...,
    cmap="cividis",
    tooltip=False,
    popup=["name", "heightroof"],
    legend=True,
)
m

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

<hr style="border: 1px solid #fdb515;" />

### ✏️ Try It Out 3: Explore the `.explore()` options

Read the [`.explore()` documentation](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.explore.html). Redraw the map using `scheme=`, `legend_kwds=`, and one other parameter of your choice. Add a comment beside each one saying what it does and how it changed the map.

In [ ]:
# your code here

<!-- END QUESTION -->

### Drawing Centroids

Before computing centroids, reproject to a projected CRS so that distances and areas are in meaningful units (feet or meters, not degrees). EPSG:2263 is NAD83 / New York Long Island, appropriate for local NYC analysis.

**Tip:** Why does CRS matter for centroid calculation? A centroid computed in geographic coordinates (degrees) will be geometrically incorrect for non-equatorial locations because degrees of longitude are not equal in length to degrees of latitude.

GeoPandas uses `set_crs()` to assign a CRS to geometries that are missing, and `to_crs()` to reproject geometries into a different CRS for analysis or mapping. Here we reproject the building footprints to EPSG:2263, then map their centroids.

In [ ]:
bldg_fpt_2263 = bldg_fpt.to_crs(2263)
bldg_fpt_2263.centroid.explore(
    tiles="Stadia.AlidadeSmooth",
    color=BERKELEY_BLUE,
    marker_kwds={"radius": 2},
)

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Coordinate Reference Systems and Measurement

<hr style="border: 1px solid #fdb515;" />

The CRS defines how coordinates in your data relate to locations on Earth. This matters the moment you want to measure anything: area, length, distance. If your data is in geographic coordinates (degrees of latitude and longitude), those measurements will be wrong.

Two broad categories:

- **Geographic CRS (GCS):** coordinates in degrees. WGS84 (EPSG:4326) is the most common. Good for storing and sharing data, bad for measuring.
- **Projected CRS (PCS):** coordinates in linear units (meters or feet). Correct for measurements, but valid only within a limited region.

The rule is simple: always reproject to a projected CRS before computing area, length, distance, or buffer.

In [ ]:
# What CRS is each version of the building data in?
print("bldg_fpt CRS:     ", bldg_fpt.crs)
print("bldg_fpt_2263 CRS:", bldg_fpt_2263.crs)

### What happens if you measure in the wrong CRS?

Let us compute building area in both CRS and compare.

In [ ]:
# Area in the geographic CRS: square degrees, which have no physical meaning
area_degrees = bldg_fpt.geometry.area
print("Area in degrees squared:")
print(area_degrees.describe().round(8))

In [ ]:
# Area in the projected CRS: EPSG:2263 uses US survey feet
area_sqft = bldg_fpt_2263.geometry.area
print("Area in square feet:")
print(area_sqft.describe().round(1))

The numbers in degrees are near-zero and uninterpretable. The numbers in square feet are actual building footprint areas.

Why square feet and not meters? EPSG:2263 (NAD83 / New York Long Island) uses US survey feet. If you want square meters, use EPSG:32618 (UTM Zone 18N) instead.

In [ ]:
# Same data in meters using UTM Zone 18N
bldg_fpt_utm = bldg_fpt.to_crs(32618)
area_sqm = bldg_fpt_utm.geometry.area
print("Area in square meters:")
print(area_sqm.describe().round(1))

### Perimeter and length

`.length` returns the perimeter for polygons and the length for lines. Units follow the CRS.

In [ ]:
# Perimeter of each building footprint in feet
bldg_fpt_2263['perimeter_ft'] = bldg_fpt_2263.geometry.length
bldg_fpt_2263['area_sqft']    = bldg_fpt_2263.geometry.area

bldg_fpt_2263[['name', 'area_sqft', 'perimeter_ft']].sort_values('area_sqft', ascending=False).head(10)

### Distance between geometries

`.distance()` computes the distance between each geometry and another geometry or GeoSeries. Units follow the CRS.

Below we find the centroid of the tallest building and measure how far every other building is from it.

In [ ]:
# Find the tallest building
tallest = bldg_fpt_2263.loc[bldg_fpt_2263["heightroof"].idxmax()]
tallest_name = tallest["name"] if pd.notna(tallest["name"]) else "(no name recorded)"
print("Tallest building:", tallest_name, "| BIN:", tallest["bin"], "| height:", tallest["heightroof"], "ft")

# Distance between every building and the tallest one, in feet
bldg_fpt_2263["dist_to_tallest_ft"] = bldg_fpt_2263.geometry.distance(tallest.geometry)
bldg_fpt_2263[["name", "dist_to_tallest_ft"]].sort_values("dist_to_tallest_ft").head(10)

<hr style="border: 1px solid #fdb515;" />

### ✏️ Try It Out 4: Total footprint area

Using `bldg_fpt_2263`, compute the total footprint area of all buildings in **acres** (1 acre = 43,560 sq ft) and store it in `total_acres`. Then find the single largest footprint and store its area in square feet in `largest_footprint_sqft`.

In [ ]:
total_acres = ...
largest_footprint_sqft = ...

print(f"Total footprint area: {total_acres:,.1f} acres")
print(f"Largest single footprint: {largest_footprint_sqft:,.0f} sq ft")

In [ ]:
grader.check("ca06s2_acres")

<hr style="border: 1px solid #fdb515;" />

### ✏️ Try It Out 5: The same area in meters

Reproject `bldg_fpt` to EPSG:32618 (UTM Zone 18N, meters), compute the total area in square meters, and convert it to **hectares** (1 hectare = 10,000 sq m). Store it in `total_hectares`.

One acre is 0.4047 hectares. Do your two answers agree? If not exactly, why might two projected CRSs give slightly different areas?

In [ ]:
bldg_fpt_32618 = ...
total_hectares = ...

print(f"Total footprint area: {total_hectares:,.2f} hectares")
print(f"From Try It Out 4:    {total_acres * 0.404686:,.2f} hectares")

In [ ]:
grader.check("ca06s2_hectares")

<hr style="border: 1px solid #fdb515;" />

### ✏️ Try It Out 6: A 50-meter buffer

Buffer each building's centroid by **50 meters** and store the result in `centroid_buffers`. Which CRS must the data be in before you buffer, and why? Plot the result.

In [ ]:
centroid_buffers = ...

ax = centroid_buffers.plot(color=BERKELEY_BLUE, alpha=0.3, edgecolor="white", figsize=(8, 8))
ax.set_title("50 m buffers around building centroids", loc="left")
ax.set_axis_off()
plt.show()

In [ ]:
grader.check("ca06s2_buffer")

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## ⭐ Optional: Inset Maps

<hr style="border: 1px solid #fdb515;" />

An inset map adds a secondary map panel inside or beside the main map. Two common uses:

- **Zoom-in inset:** the main map shows a wide area; the inset zooms into one small region
- **Locator inset:** the main map shows a small region; the inset shows where that region sits in the larger context



We will use `mpl_toolkits` with `mark_inset()`

`mark_inset()` draws connector lines between the inset panel and the corresponding extent on the parent axis. Both axes must use the same coordinate system (i.e., the same CRS/units).

To read more and learn other methods of achieving this checkout [this example](https://geopandas.org/en/stable/gallery/inset_map.html).

In [ ]:
nyc_neighs = gpd.read_parquet("data/raw/nyc_neighborhoods_small.parquet")

ax = nyc_neighs.plot(color=BERKELEY_BLUE, edgecolor="white", linewidth=0.4, figsize=(8, 8))
ax.set_title("NYC neighborhoods", loc="left")
ax.set_axis_off()
plt.show()

In [ ]:
# Manhattan sub-boundary codes fall between 300 and 400.
manh_mask = (nyc_neighs["bor_subb"] > 300) & (nyc_neighs["bor_subb"] < 400)
manh_neigh = nyc_neighs.loc[manh_mask].copy()

ax = manh_neigh.plot(color=BERKELEY_BLUE, edgecolor="white", linewidth=0.4, figsize=(6, 8))
ax.set_title("Manhattan neighborhoods", loc="left")
ax.set_axis_off()
plt.show()

In [ ]:
from mpl_toolkits.axes_grid1 import inset_locator

# Main map: all Manhattan neighborhoods
fig, ax = plt.subplots(figsize=(8, 8))
manh_neigh.plot(ax=ax, color="#d9d9d9", edgecolor="white", linewidth=0.5)
ax.set_title("Upper West Side, located within Manhattan", loc="left")
ax.set_axis_off()

# Choose a neighborhood to zoom into
target_neigh = manh_neigh[manh_neigh["name"] == "Upper West Side"]
minx, miny, maxx, maxy = target_neigh.buffer(300).total_bounds

# Inset axis: upper right, 1.5 x 1.5 inches
iax = inset_locator.inset_axes(
    ax, width=1.5, height=1.5, loc="upper right",
    borderpad=1, axes_kwargs={"xticks": [], "yticks": []},
)
manh_neigh.plot(ax=iax, color="#d9d9d9", edgecolor="white", linewidth=0.5)
target_neigh.plot(ax=iax, color=BERKELEY_GOLD)
iax.set_xlim(minx, maxx)
iax.set_ylim(miny, maxy)

# Connect inset to parent. Corners: 1 = upper right, 2 = upper left, 3 = lower left, 4 = lower right
inset_locator.mark_inset(ax, iax, loc1=2, loc2=4, linewidth=0.8, edgecolor=BERKELEY_BLUE)

plt.show()

**What happened here?**

`inset_axes()` creates a child axis positioned within the parent axis. `mark_inset()` draws the connection lines. The `loc1` and `loc2` arguments specify which corners of the inset connect to which corners of the highlighted region on the main map (1=upper right, 2=upper left, 3=lower left, 4=lower right).

Why do both axes need the same CRS? `mark_inset()` computes the connection lines using coordinates. If the CRS differs, the lines point to the wrong location.

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Summary

<hr style="border: 1px solid #fdb515;" />

| Task | Pattern |
|---|---|
| Read GeoParquet | `gpd.read_parquet("path.parquet")` |
| Read a Shapefile, GeoJSON, or URL | `gpd.read_file("path or url")` |
| Build points from a CSV | `gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.lon, df.lat), crs=4326)` |
| Filter at read time | `gpd.read_file("path", bbox=bbox)` |
| Inspect the geometry column | `.geometry` returns a `GeoSeries` |
| Check and change the CRS | `.crs`, `.to_crs(epsg)`, `.set_crs(epsg)` |
| Measure | `.area`, `.length`, `.distance()`, `.buffer()`, in the CRS's units |
| Write | `.to_file("path.geojson")`, `.to_parquet("path.parquet")` |
| Map | `.plot("column", legend=True)`, `.explore("column", tiles="Stadia.AlidadeSmooth")` |